# Cálculo 1 — Visualizações Interativas

Este notebook reúne **plots didáticos** para estudar os três pilares do Cálculo 1:

1. **Limites** — comportamento de $f(x)$ quando $x \to a$ (incluindo limites laterais).
2. **Derivadas** — reta tangente e a curva da derivada $f'(x)$.
3. **Integrais** — área sob a curva e a soma de Riemann pelo **ponto médio**.

> **Como usar:** edite a variável `expr_str` na célula de configuração abaixo com a função desejada (apenas uma variável `x`) e reexecute as células. Cada seção também possui parâmetros próprios (ponto de análise, intervalo, número de partições) que você pode ajustar.

**Exemplos de funções:** `sin(x)/x`, `x**2 - 2*x`, `exp(-x**2)`, `1/x`, `tan(x)`, `sqrt(x)`, `log(x)`.


---

## Como usar este notebook

1. **Troque a função:** edite `expr_str` na célula de **configuração** (ex.: `"x**2 - 2*x"`, `"exp(-x**2)"`, `"1/x"`, `"tan(x)"`) e reexecute a partir dali.
2. **Ajuste os parâmetros de cada seção:**
   - **Limites** → `a_lim` (ponto de análise) e `janela`.
   - **Derivadas** → `a_der` (ponto de tangência) e o intervalo `x_ini, x_fim`.
   - **Integrais** → limites `a_int, b_int` e o número de retângulos `N` (aumente para melhorar a aproximação).
3. **Sintaxe da função (SymPy):** use `**` para potência (`x**3`), e funções como `sin`, `cos`, `tan`, `exp`, `log`, `sqrt`, `Abs`. A constante $\pi$ é `pi` e $e$ é `E`.

> As descontinuidades (ex.: `1/x`, `tan(x)`) são tratadas automaticamente: a linha é "quebrada" nas assíntotas para não poluir o gráfico.


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display

matplotlib.style.use('ggplot')

# =====================================================================
#  FUNÇÃO PERSONALIZADA  ->  edite a string abaixo (apenas variável x)
# =====================================================================
expr_str = "sin(x)/x"
# ---------------------------------------------------------------------

x = sp.symbols('x')

# Converte o texto em uma expressão simbólica do SymPy
f_sym = sp.sympify(expr_str)

# Versão numérica (vetorizada com NumPy) para os gráficos
f_num = sp.lambdify(x, f_sym, 'numpy')

print("Função definida:")
display(sp.Eq(sp.Function('f')(x), f_sym))


def avalia(func, xs, limite=1e3):
    """Avalia `func` em `xs` de forma segura para plotagem.

    - Suprime avisos de divisão/valor inválido (1/x, tan(x), log(x)...).
    - Garante formato de array (funções constantes viram vetor).
    - Substitui por NaN os pontos com |y| muito grande ou inválidos,
      para que o Matplotlib "quebre" a linha nas assíntotas em vez de
      desenhar traços verticais espúrios.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        ys = func(xs)
    ys = np.asarray(ys, dtype=float) * np.ones_like(xs)  # broadcast p/ constantes
    ys[np.abs(ys) > limite] = np.nan
    return ys


def nova_figura(titulo):
    """Cria uma figura padronizada no estilo dos demais notebooks."""
    plt.figure(figsize=(10, 5), dpi=100, facecolor='w', edgecolor='k')
    plt.title(titulo, fontsize=13, fontweight='bold')
    plt.axhline(y=0, color='0.4', linewidth=1)   # eixo x
    plt.axvline(x=0, color='0.4', linewidth=1)   # eixo y


print("\nConfiguração concluída. Edite 'expr_str' acima para trocar a função.")


## 1. Limites

O **limite** descreve para qual valor $f(x)$ se aproxima quando $x$ tende a um ponto $a$:

$$\lim_{x \to a} f(x) = L$$

O gráfico abaixo mostra a função em torno de $x = a$, destacando as aproximações **pela esquerda** ($x \to a^-$) e **pela direita** ($x \to a^+$). Quando os limites laterais coincidem, o limite existe.

> **Parâmetro:** ajuste `a_lim` (o ponto de análise) na célula abaixo.


In [ ]:
# -------- Parâmetro do limite --------
a_lim = 0.0        # ponto onde x -> a
janela = 4.0       # largura da janela em torno de 'a' que será plotada
# -------------------------------------

# Limites simbólicos (bilateral e laterais)
L_bi  = sp.limit(f_sym, x, a_lim)
L_esq = sp.limit(f_sym, x, a_lim, dir='-')
L_dir = sp.limit(f_sym, x, a_lim, dir='+')

print(f"lim(x -> {a_lim})  f(x) = {L_bi}")
print(f"lim(x -> {a_lim}-) f(x) = {L_esq}   (pela esquerda)")
print(f"lim(x -> {a_lim}+) f(x) = {L_dir}   (pela direita)")

# Amostras à esquerda e à direita (evitam o ponto exato x = a)
xe = np.linspace(a_lim - janela, a_lim, 400, endpoint=False)
xd = np.linspace(a_lim + janela, a_lim, 400, endpoint=False)[::-1]
ye = avalia(f_num, xe)
yd = avalia(f_num, xd)

nova_figura(f"Limite de f(x) quando x → {a_lim}")
plt.plot(xe, ye, color='#1f77b4', linewidth=2, label='aproximação pela esquerda (x → a⁻)')
plt.plot(xd, yd, color='#d62728', linewidth=2, label='aproximação pela direita (x → a⁺)')
plt.axvline(x=a_lim, color='green', linestyle='--', linewidth=1.5, alpha=0.8, label=f'x = a = {a_lim}')

# Marca o valor do limite bilateral, quando finito
if L_bi.is_finite:
    Lval = float(L_bi)
    plt.plot(a_lim, Lval, 'o', color='green', markersize=9, markerfacecolor='white', zorder=5)
    plt.annotate(f'L = {Lval:.4g}', xy=(a_lim, Lval), xytext=(20, 20),
                 textcoords='offset points', fontsize=11,
                 arrowprops=dict(arrowstyle='->', color='green'),
                 bbox=dict(boxstyle="round,pad=0.3", fc="lightgreen", alpha=0.7))

plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True, alpha=0.5)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()


## 2. Derivadas

A **derivada** $f'(x)$ mede a taxa de variação instantânea de $f$ e corresponde à **inclinação da reta tangente** ao gráfico em cada ponto:

$$f'(a) = \lim_{h \to 0} \frac{f(a+h) - f(a)}{h}$$

Abaixo geramos **dois gráficos separados**:

1. $f(x)$ com a **reta tangente** no ponto $x = a$ (inclinação $= f'(a)$);
2. a curva da **derivada** $f'(x)$, onde $f'(x) = 0$ indica pontos de máximo/mínimo de $f$.

> **Parâmetro:** ajuste `a_der` (ponto de tangência) na célula abaixo.


In [ ]:
# -------- Parâmetros da derivada --------
a_der = 1.0                  # ponto de tangência
x_ini, x_fim = -5.0, 5.0     # intervalo de plotagem
# ----------------------------------------

# Derivada simbólica
df_sym = sp.diff(f_sym, x)
df_num = sp.lambdify(x, df_sym, 'numpy')

print("Derivada:")
display(sp.Eq(sp.Derivative(sp.Function('f')(x), x), df_sym))

# Valores no ponto de tangência
fa  = float(f_num(a_der))
dfa = float(df_num(a_der))
print(f"\nf({a_der})  = {fa:.4g}")
print(f"f'({a_der}) = {dfa:.4g}   (inclinação da tangente)")

xs = np.linspace(x_ini, x_fim, 800)
ys = avalia(f_num, xs)

# ---- Gráfico 1: f(x) + reta tangente ----
reta_tangente = fa + dfa * (xs - a_der)   # y = f(a) + f'(a)(x - a)

nova_figura(f"f(x) e a reta tangente em x = {a_der}")
plt.plot(xs, ys, color='#1f77b4', linewidth=2, label='f(x)')
plt.plot(xs, reta_tangente, color='#ff7f0e', linestyle='--', linewidth=2,
         label=f"reta tangente (inclinação = {dfa:.3g})")
plt.plot(a_der, fa, 'o', color='black', markersize=8, zorder=5)
plt.annotate(f'({a_der}, {fa:.3g})', xy=(a_der, fa), xytext=(15, 15),
             textcoords='offset points', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='black'))
plt.xlabel('x')
plt.ylabel('f(x)')
plt.ylim(np.nanmin(ys) - 1, np.nanmax(ys) + 1)
plt.grid(True, alpha=0.5)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

# ---- Gráfico 2: curva da derivada f'(x) ----
dys = avalia(df_num, xs)

nova_figura("Curva da derivada f'(x)")
plt.plot(xs, dys, color='#2ca02c', linewidth=2, label="f'(x)")
plt.plot(a_der, dfa, 'o', color='black', markersize=8, zorder=5,)
plt.xlabel('x')
plt.ylabel("f'(x)")
plt.grid(True, alpha=0.5)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()



## 3. Integrais

A **integral definida** representa a **área com sinal** entre a curva $f(x)$ e o eixo $x$ no intervalo $[a, b]$:

$$\int_{a}^{b} f(x)\,dx$$

Numericamente, essa área pode ser aproximada pela **Soma de Riemann pelo ponto médio**, dividindo $[a, b]$ em $N$ retângulos de largura $\Delta x = \dfrac{b-a}{N}$:

$$\int_{a}^{b} f(x)\,dx \approx \sum_{i=1}^{N} f(m_i)\,\Delta x, \qquad m_i = a + \left(i - \tfrac{1}{2}\right)\Delta x$$

O gráfico mostra a área exata sombreada e os retângulos do ponto médio. Aumentar `N` aproxima a soma do valor exato.

> **Parâmetros:** ajuste os limites `a_int`, `b_int` e o número de partições `N` na célula abaixo.


In [ ]:
# -------- Parâmetros da integral --------
a_int, b_int = 0.0, 3.0    # limites de integração [a, b]
N = 10                     # número de retângulos (partições)
# ----------------------------------------

# Integral definida exata (simbólica) com fallback numérico
integral_exata = sp.integrate(f_sym, (x, a_int, b_int))
try:
    valor_exato = float(integral_exata.evalf())
    if not np.isfinite(valor_exato):
        raise ValueError
except (TypeError, ValueError):
    # fallback numérico caso não haja forma fechada
    xx = np.linspace(a_int, b_int, 20001)
    valor_exato = float(np.trapezoid(avalia(f_num, xx), xx))
    print("(valor exato obtido por integração numérica)")

print("Integral indefinida:")
display(sp.Eq(sp.Integral(f_sym, x), sp.integrate(f_sym, x)))
print(f"\nValor exato de  ∫[{a_int}, {b_int}] f(x) dx = {valor_exato:.6f}")

# Soma de Riemann pelo PONTO MÉDIO
dx = (b_int - a_int) / N
bordas = np.linspace(a_int, b_int, N + 1)
pontos_medios = (bordas[:-1] + bordas[1:]) / 2      # m_i
alturas = avalia(f_num, pontos_medios)
soma_riemann = float(np.nansum(alturas * dx))

erro = abs(valor_exato - soma_riemann)
print(f"Soma de Riemann (ponto médio, N = {N}) = {soma_riemann:.6f}")
print(f"Erro absoluto vs. valor exato          = {erro:.6f}")

# Curva suave para o contorno da área
xs = np.linspace(a_int, b_int, 800)
ys = avalia(f_num, xs)

nova_figura(f"Integral de f(x) em [{a_int}, {b_int}]  —  Soma do ponto médio (N = {N})")
plt.plot(xs, ys, color='#1f77b4', linewidth=2, zorder=3, label='f(x)')

# Área exata sombreada
plt.fill_between(xs, ys, 0, alpha=0.20, color='#1f77b4',
                 label=f'área exata ≈ {valor_exato:.4g}')

# Retângulos do ponto médio
plt.bar(pontos_medios, alturas, width=dx, alpha=0.35, color='#ff7f0e',
        edgecolor='#d62728', linewidth=1.0, align='center',
        label=f'retângulos (ponto médio) ≈ {soma_riemann:.4g}')
plt.plot(pontos_medios, alturas, 'o', color='#d62728', markersize=4, zorder=4)

plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True, alpha=0.5)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()


## 4. Aplicação: Volume de um Sólido de Revolução (Coordenadas Cilíndricas)

Uma aplicação clássica da integral é o cálculo do **volume de um sólido de revolução**. Tomamos um **perfil** (geratriz) $R(x)$ — por exemplo $R(x) = -x^2 + 10$ — que representa a **distância ao eixo de rotação** (aqui, o eixo $x$), e o giramos em torno desse eixo, gerando um sólido de **aspecto cilíndrico** (barril).

Em **coordenadas cilíndricas** $(r, \theta, x)$, com o eixo $x$ como eixo de revolução, o elemento de volume é

$$dV = r \, dr \, d\theta \, dx$$

onde o fator $r$ é o **jacobiano** da transformação. O volume total é a integral tripla

$$V = \int_{a}^{b} \int_{0}^{\Theta} \int_{0}^{R(x)} r \, dr \, d\theta \, dx = \frac{\Theta}{2}\int_{a}^{b} \big[R(x)\big]^2\, dx$$

Os limites $a$ e $b$ são as **extremidades** do sólido (onde $R(x) = 0$) e $\Theta$ é o ângulo de revolução.

> 💡 **Sobre os 180° vs. 360°:** como o perfil é **simétrico** em relação ao eixo, girar a seção completa (de $-R(x)$ a $+R(x)$) por **180°** varre exatamente o mesmo sólido que girar apenas o raio ($0$ a $R(x)$) por **360°**. Na formulação cilíndrica padrão usamos $r \in [0, R(x)]$ e $\theta \in [0, 2\pi]$; ajuste `angulo_graus` abaixo para experimentar (use `180` para ver meia-revolução).


In [ ]:
# ============================================================
#  VOLUME DE UM SÓLIDO DE REVOLUÇÃO  (coordenadas cilíndricas)
# ============================================================
# Obs.: no matplotlib >= 3.1 a projeção '3d' é registrada automaticamente.

# -------- Perfil e parâmetros (edite à vontade) --------
perfil_str = "-x**2 + 10"   # R(x): distância ao eixo de revolução (eixo x)
angulo_graus = 360          # ângulo de revolução (use 180 p/ meia-volta)
# -------------------------------------------------------

# Perfil simbólico e numérico
R_sym = sp.sympify(perfil_str)
R_num = sp.lambdify(x, R_sym, 'numpy')

# Extremidades: pontos onde o perfil toca o eixo, R(x) = 0
raizes = [s for s in sp.solve(sp.Eq(R_sym, 0), x) if s.is_real]
if len(raizes) >= 2:
    raizes = sorted(raizes, key=lambda s: float(s))
    a_sym, b_sym = raizes[0], raizes[-1]
else:
    a_sym, b_sym = sp.Integer(-3), sp.Integer(3)   # ajuste manual se R(x) não cruzar o eixo
    print("Perfil não cruza o eixo em 2 pontos; usando limites padrão [-3, 3].")
a_vol, b_vol = float(a_sym), float(b_sym)

# ---- Volume por integral tripla em coordenadas cilíndricas ----
# dV = r dr dθ dx   ->   V = ∫∫∫ r dr dθ dx
r, th = sp.symbols('r theta', nonnegative=True)
Theta_sym = sp.Rational(angulo_graus, 180) * sp.pi     # ângulo em radianos (exato)

V_int = sp.Integral(r, (r, 0, R_sym), (th, 0, Theta_sym), (x, a_sym, b_sym))
V_sym = V_int.doit()
V_val = float(V_sym)

print(f"Perfil R(x) = {R_sym}   |   extremidades: x ∈ [{a_vol:.3f}, {b_vol:.3f}]")
print("Integral de volume em coordenadas cilíndricas:")
display(sp.Eq(sp.Symbol('V'), V_int))
display(sp.Eq(sp.Symbol('V'), V_sym))
print(f"\nVolume ≈ {V_val:.4f} unidades³   (θ = {angulo_graus}°)")

# ---- Gráfico 1: perfil (geratriz) que será girado ----
xa = np.linspace(a_vol, b_vol, 400)
Ra = np.clip(R_num(xa), 0, None)

nova_figura(f"Perfil (geratriz) do sólido — R(x) = {R_sym}")
plt.plot(xa, Ra, color='#1f77b4', linewidth=2, label='R(x) — perfil')
plt.plot(xa, -Ra, color='#1f77b4', linewidth=2, linestyle='--', label='-R(x) — espelho')
plt.fill_between(xa, Ra, -Ra, alpha=0.15, color='#1f77b4',
                 label='seção transversal girada em torno do eixo x')
plt.xlabel('x (eixo de revolução)')
plt.ylabel('raio')
plt.grid(True, alpha=0.5)
plt.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

# ---- Gráfico 2: sólido de revolução em 3D ----
theta_max = np.deg2rad(angulo_graus)
xg = np.linspace(a_vol, b_vol, 80)
tg = np.linspace(0, theta_max, 80)
Xg, Tg = np.meshgrid(xg, tg)
Rg = np.clip(R_num(Xg), 0, None)   # raio não-negativo
Yg = Rg * np.cos(Tg)               # y = R(x)·cos(θ)
Zg = Rg * np.sin(Tg)               # z = R(x)·sin(θ)

fig = plt.figure(figsize=(9, 7), dpi=100, facecolor='w')
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(Xg, Yg, Zg, cmap='viridis', alpha=0.9,
                       rstride=2, cstride=2, linewidth=0, antialiased=True)
ax.set_title(f"Sólido de revolução  (θ = {angulo_graus}°)  —  V ≈ {V_val:.2f}",
             fontsize=12, fontweight='bold')
ax.set_xlabel('x (eixo)')
ax.set_ylabel('y')
ax.set_zlabel('z')
# Proporção realista (aspecto de barril/cilindro)
raio_max = float(np.nanmax(Rg))
ax.set_box_aspect((b_vol - a_vol, 2 * raio_max, 2 * raio_max))
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=12, label='raio R(x)')
plt.tight_layout()
plt.show()
